# Capstone — Structured Content Archetype Clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shiva-sn/ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook consolidates the research question, data contract, leakage controls, W05 clustering result, W06 validation framing, and W07 action playbook into one paper-ready capstone record.

The analysis is descriptive and decision-support oriented: it identifies recurring observed content patterns and helps prioritize human review. It does not establish causal effects or predict future search performance.

## 1. Question

**Research question:** What performance archetypes exist across the content inventory?

**Decision supported:** Which types of content should a content/SEO team review first, and what kind of review may be appropriate?

The analysis groups content items with similar observed search demand, visibility, freshness, content size, and engagement characteristics. The goal is to replace an unordered large inventory with a small number of interpretable archetypes for prioritization.

In [ ]:
research_question = "What performance archetypes exist across the content inventory?"
decision_supported = "Prioritize human review of content using observed archetype patterns."
task_type = "unsupervised clustering"

print("Research question:", research_question)
print("Decision supported:", decision_supported)
print("Task type:", task_type)
assert task_type == "unsupervised clustering"

## 2. Data

The capstone uses the anonymized structured content dataset and the W05 content-level modeling output. The core modeling grain is **one row per content item**. The W05 pipeline combines content attributes, aggregated daily performance, and aggregated 90-day search information.

The main analysis window is a rolling 90-day performance window ending at the latest available performance date in the W05 run.

**Core features:** `search_volume`, `word_count`, `content_age_days`, `days_since_update`, `impressions_90d`, `ctr_90d`, `avg_position_90d`, `engagement_rate`.

**Excluded from clustering:** identifiers, query breadth, future/trend/label-like fields, provider/model metadata, and any client-name/domain/URL fields. `avg_position_90d = 0` is treated as missing because it represents no position data.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("..")
OUTPUT_DIR = BASE / "outputs"
CANDIDATES = [
    OUTPUT_DIR / "content_archetypes_clustered.parquet",
    OUTPUT_DIR / "content_level_model_dataset.parquet",
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Run W05 first and keep its parquet output under work/outputs/.")

df = pd.read_parquet(DATA_PATH)
core_features = [
    "search_volume", "word_count", "content_age_days", "days_since_update",
    "impressions_90d", "ctr_90d", "avg_position_90d", "engagement_rate"
]
required = ["content_hash_id"] + core_features
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required fields: {missing}")

print("Loaded:", DATA_PATH.resolve())
print("Rows:", len(df))
print("Unique content:", df["content_hash_id"].nunique())
print("Core features:", len(core_features))
print("One row per content:", df["content_hash_id"].nunique() == len(df))

assert df["content_hash_id"].nunique() == len(df)

## 3. Methodology

This is an unsupervised clustering task because no ground-truth archetype label exists. W05 used K-Means with K=3 after feature preparation: convert position zero to missing, median-impute numeric missingness, apply `log1p` to heavily skewed volume/count variables (`search_volume`, `word_count`, `impressions_90d`), then use `RobustScaler`.

The core feature set contains eight numeric signals. The simple baseline used three volume/size features. Model selection considered silhouette separation together with cluster balance rather than maximizing one metric blindly. W06's honest validation design uses grouped-by-client evaluation: preprocessing is fit on development clients and held-out clients are assigned to development-fitted centroids.

No target label is predicted, and no future/trend label is used in the clustering matrix.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

X_raw = df[core_features].copy()
zero_position_n = int((X_raw["avg_position_90d"] == 0).sum())
X_raw["avg_position_90d"] = X_raw["avg_position_90d"].replace(0, np.nan)

imputer = SimpleImputer(strategy="median")
X_imp = pd.DataFrame(imputer.fit_transform(X_raw), columns=core_features, index=df.index)

log_features = ["search_volume", "word_count", "impressions_90d"]
for col in log_features:
    X_imp[col] = np.log1p(X_imp[col].clip(lower=0))

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imp)

print("K-Means K:", int(df["cluster"].nunique()) if "cluster" in df.columns else 3)
print("Position zeros converted to missing:", zero_position_n)
print("Missing values after imputation:", int(X_imp.isna().sum().sum()))
print("Scaled matrix shape:", X_scaled.shape)
assert int(X_imp.isna().sum().sum()) == 0
assert X_scaled.shape[1] == 8

## 4. Results (vs baseline)

The final W05 solution uses **K=3**. The reference W05 full-population silhouette reported in the validation notebook is **0.8414**. This is an internal separation metric, not accuracy and not a measure of business impact.

The three observed archetypes are:

1. **Low-Visibility Established Content** → Improve
2. **Stale Underperforming Content** → Rewrite
3. **High-CTR Efficient Niche Content** → Protect

The third cluster is very small (about 0.1% in the W05 interpretation), so it should be treated as a rare, narrow pattern rather than a broad portfolio segment.

In [ ]:
W05_K = 3
W05_SILHOUETTE_REFERENCE = 0.8414

cluster_sizes = df["cluster"].value_counts().sort_index() if "cluster" in df.columns else pd.Series(dtype=int)
cluster_share = (cluster_sizes / len(df) * 100).round(3) if len(cluster_sizes) else pd.Series(dtype=float)

results_table = pd.DataFrame({
    "metric": ["W05 K", "W05 reference silhouette", "content items", "clusters"],
    "value": [W05_K, W05_SILHOUETTE_REFERENCE, len(df), df["cluster"].nunique() if "cluster" in df.columns else np.nan]
})
display(results_table)

if len(cluster_sizes):
    display(pd.DataFrame({"cluster_n": cluster_sizes, "share_pct": cluster_share}))

### Archetype interpretation

Cluster 0 is treated as a low-visibility established group: its main decision is targeted improvement review. Cluster 1 is a stale/underperforming group: it is a candidate for deeper freshness/content review. Cluster 2 is a rare high-efficiency niche pattern: protect from unnecessary changes, but manually verify because the group is small. These labels are interpretations of observed profiles, not ground-truth classes.

In [ ]:
archetype_map = {
    0: {"archetype": "Low-Visibility Established Content", "action": "Improve"},
    1: {"archetype": "Stale Underperforming Content", "action": "Rewrite"},
    2: {"archetype": "High-CTR Efficient Niche Content", "action": "Protect"},
}

if "cluster" in df.columns:
    observed = sorted(df["cluster"].dropna().astype(int).unique())
    assert all(c in archetype_map for c in observed)
    print("Observed cluster → action mapping:")
    for c in observed:
        print(c, "→", archetype_map[c])

## 5. Limitations

- Clustering is unsupervised: there is no ground-truth archetype label.
- Silhouette is an internal separation metric; it is not accuracy or business impact.
- The 90-day observation window is a snapshot and may not represent every content lifecycle.
- Median imputation can influence cluster boundaries when missingness is substantial.
- Grouped-by-client validation tests generalization to unseen clients, not future time periods.
- Rare clusters should be interpreted as narrow observed patterns.
- The structured dataset does not contain article text, so this is metric-based rather than semantic clustering.
- The analysis does not prove that rewriting, updating, protecting, or monitoring content will cause better rankings or traffic.

In [ ]:
limitations = [
    "No ground-truth archetype label exists.",
    "Silhouette is an internal metric, not business impact.",
    "The 90-day window is a snapshot.",
    "Imputation can influence cluster boundaries.",
    "Unseen-client validation is not future-time validation.",
    "Rare clusters need manual verification.",
    "The analysis is metric-based, not semantic.",
    "No causal effect of content changes is established.",
]
for item in limitations:
    print("-", item)

## 6. Ranked recommendations

The recommendation layer follows the W07 playbook. It ranks review work; it does not automate content changes. Ambiguous assignments, material missingness, and rare protect clusters are routed to **Review** before action.

In [ ]:
queue_path = OUTPUT_DIR / "content_action_queue.csv"
summary_path = OUTPUT_DIR / "content_action_summary.csv"

if queue_path.exists():
    queue = pd.read_csv(queue_path)
    print("Loaded W07 queue:", queue_path.resolve())
    display(queue.head(20))
else:
    # Fallback: create a compact cluster-level recommendation table from the current data.
    fallback = pd.DataFrame([
        [0, archetype_map[0]["archetype"], "Improve", "Review low-visibility established patterns and prioritize targeted optimization."],
        [1, archetype_map[1]["archetype"], "Rewrite", "Review stale/underperforming content before deciding whether deeper revision is appropriate."],
        [2, archetype_map[2]["archetype"], "Protect", "Avoid unnecessary changes; manually verify the rare high-efficiency pattern."],
    ], columns=["cluster", "archetype", "action", "decision_note"])
    display(fallback)

if summary_path.exists():
    print("\nW07 action summary:")
    display(pd.read_csv(summary_path))

## 7. Artifacts the paper embeds

The paper-facing artifacts are the clustered content dataset, the ranked action queue, the action summary, and the concise playbook markdown. This notebook also creates a compact results table that can be reused in a manuscript.

In [ ]:
artifact_paths = [
    OUTPUT_DIR / "content_archetypes_clustered.parquet",
    OUTPUT_DIR / "content_level_model_dataset.parquet",
    OUTPUT_DIR / "content_action_queue.csv",
    OUTPUT_DIR / "content_action_summary.csv",
    OUTPUT_DIR / "content_action_playbook.md",
]

artifact_status = pd.DataFrame({
    "artifact": [p.name for p in artifact_paths],
    "exists": [p.exists() for p in artifact_paths],
})
display(artifact_status)

# A paper-ready compact table, generated from available cluster assignments.
if "cluster" in df.columns:
    paper_table = (
        df.groupby("cluster")[core_features]
        .median()
        .round(2)
        .reset_index()
    )
    paper_table["archetype"] = paper_table["cluster"].map(lambda c: archetype_map.get(int(c), {}).get("archetype", "Review"))
    paper_table["recommended_action"] = paper_table["cluster"].map(lambda c: archetype_map.get(int(c), {}).get("action", "Review"))
    display(paper_table)

## Closing: 5-minute demo outline

**Minute 1:** State the question and explain why clustering is used instead of predicting a future label.

**Minute 2:** Show the one-row-per-content grain and the eight core observed features.

**Minute 3:** Show the K=3 archetypes and the silhouette result, then emphasize that silhouette is an internal metric.

**Minute 4:** Show the W07 ranked action queue and explain the human-review gates.

**Minute 5:** State the limitations clearly: snapshot data, unsupervised labels, rare cluster, imputation effects, and no causal claims.

## Social-post cut

Built an unsupervised content-archetype clustering workflow that groups content pages by observed search demand, visibility, freshness, content size, and engagement signals. The final K=3 solution turns a large content inventory into a small set of interpretable patterns and a ranked human-review queue. The result is decision-support—not a claim that a specific content change will cause higher traffic or rankings.

## Employer-facing summary

I built an end-to-end unsupervised ML workflow for structured content archetype clustering, including a data contract, feature/leakage review, preprocessing, K-Means clustering, internal validation, and an action playbook. The workflow emphasizes reproducibility, explicit exclusion of future/identifier leakage, and human review of ambiguous or rare cases. The final output converts model structure into a practical prioritization artifact without overstating what the data can prove.

## Acknowledgments & data credit

Data source / internship warehouse credit: **FlyRank** — https://flyrank.ai

This capstone uses an anonymized structured dataset and reports only aggregate/hashed, public-safe analysis outputs.

## Self-check

Before submission, confirm that the notebook runs top-to-bottom on a fresh runtime, all required W05/W07 output artifacts are present, and the deployed paper contains the complete 9-section structure including the Abstract at the top and Acknowledgments & data credit at the bottom.

The capstone language should remain limited to **observed**, **measured**, **directional**, and **decision-support** claims.

In [ ]:
checks = [
    ("Data loaded", len(df) > 0),
    ("One row per content", df["content_hash_id"].nunique() == len(df)),
    ("Eight core features present", len(core_features) == 8 and all(c in df.columns for c in core_features)),
    ("No identifiers in core features", not any(c in core_features for c in ["client_hash_id", "content_hash_id"])),
    ("No future/trend/label names in core features", not any(k in c.lower() for c in core_features for k in ["future", "trend", "label", "target", "outcome"])),
    ("No query feature in core features", not any("query" in c.lower() for c in core_features)),
    ("No remaining missing values in prepared matrix", int(X_imp.isna().sum().sum()) == 0),
]

check_df = pd.DataFrame(checks, columns=["check", "passed"])
display(check_df)

if not check_df["passed"].all():
    raise AssertionError("One or more capstone self-checks failed.")

print("All capstone automated checks PASS.")